In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

from weather_shift.data import get_pairs
from weather_shift.metrics import evaluate_uncertainty
from weather_shift.model import predict, predict_flip, predict_mc

name = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
processor = SegformerImageProcessor.from_pretrained(name)
model = SegformerForSemanticSegmentation.from_pretrained(name).eval()

Loading weights: 100%|██████████| 208/208 [00:00<?, ?it/s]


In [ ]:
# check the model's dropout

found = Counter()
for _, module in model.named_modules():
    kind = type(module).__name__
    if "Dropout" in kind or "DropPath" in kind:
        p = getattr(module, "p", getattr(module, "drop_prob", None))
        found[(kind, round(float(p), 3))] += 1
for (kind, p), n in sorted(found.items()):
    print(f"{kind:20s} p={p:<6} x{n}")

Dropout              p=0.0    x16
Dropout              p=0.1    x1
SegformerDropPath    p=0.014  x1
SegformerDropPath    p=0.029  x1
SegformerDropPath    p=0.043  x1
SegformerDropPath    p=0.057  x1
SegformerDropPath    p=0.071  x1
SegformerDropPath    p=0.086  x1
SegformerDropPath    p=0.1    x1


In [8]:
night = get_pairs("../data", "acdc", "night")
image = Image.open(night[0][0]).convert("RGB")
pred1, _ = predict_mc(image, processor, model, n_samples=1)
pred2, _ = predict_mc(image, processor, model, n_samples=1)
print("pixels that changed between two runs:", round(float((pred1 != pred2).mean()), 4))

pixels that changed between two runs: 0.0235


In [9]:
_, conf_plain = predict(image, processor, model)
_, conf_mc = predict_mc(image, processor, model, n_samples=8)
print("mean confidence plain :", round(float(conf_plain.mean()), 3))
print("mean confidence MC :", round(float(conf_mc.mean()), 3))

mean confidence plain : 0.815
mean confidence MC : 0.774


In [ ]:
for method, fn in [("plain", predict), ("MC", predict_mc)]:
    summary, _, _ = evaluate_uncertainty(night, processor, model, predict_fn= fn)
    print(method, {k : round(float(v), 3) for k, v in summary.items()})

100%|██████████| 106/106 [00:58<00:00,  1.81it/s]


plain {'ECE': 0.277, 'AUROC': 0.742, 'confident_mistakes': 0.473}


100%|██████████| 106/106 [03:35<00:00,  2.03s/it]


MC {'ECE': 0.217, 'AUROC': 0.723, 'confident_mistakes': 0.283}


In [14]:
sets = {"clear": get_pairs("../data", "cityscapes")}
for cond in ["fog", "night", "rain", "snow"]:
    sets[cond] = get_pairs("../data", "acdc", cond)
methods = {"plain": predict, "flip": predict_flip, "mc": predict_mc}
class_names = [model.config.id2label[c] for c in range(19)]

rows, per_class = [], {}
for cond, pairs in sets.items():
    for method, fn in methods.items():
        summary, wrong_c, conf_wrong_c = evaluate_uncertainty(
            pairs, processor, model, predict_fn=fn
        )
        rows.append({"condition": cond, "method": method, **summary})
        per_class[f"{cond}_{method}"] = conf_wrong_c / np.maximum(wrong_c, 1)
        pd.DataFrame(rows).to_csv("../reports/week4_results.csv", index=False)

pd.DataFrame(per_class, index=class_names).to_csv(
    "../reports/week4_confident_mistakes_per_class.csv"
)
print(pd.DataFrame(rows).round(3))

100%|██████████| 100/100 [03:31<00:00,  2.11s/it]


   condition method    ECE  AUROC  confident_mistakes  confident_correct
0      clear  plain  0.014  0.926               0.175              0.874
1      clear   flip  0.009  0.929               0.153              0.867
2      clear     mc  0.004  0.928               0.110              0.834
3        fog  plain  0.025  0.903               0.273              0.884
4        fog   flip  0.020  0.907               0.242              0.875
5        fog     mc  0.008  0.905               0.174              0.830
6      night  plain  0.277  0.742               0.473              0.785
7      night   flip  0.256  0.739               0.425              0.747
8      night     mc  0.216  0.733               0.273              0.618
9       rain  plain  0.052  0.883               0.250              0.846
10      rain   flip  0.041  0.889               0.206              0.830
11      rain     mc  0.021  0.898               0.120              0.776
12      snow  plain  0.080  0.826               0.3

In [5]:
pc = pd.read_csv("../reports/week4_confident_mistakes_per_class.csv", index_col=0)
night_cols = ["night_plain", "night_flip", "night_mc"]
print(pc[night_cols].sort_values("night_mc", ascending=False).round(2).head(8))

              night_plain  night_flip  night_mc
sky                  0.58        0.54      0.37
sidewalk             0.58        0.54      0.32
fence                0.43        0.37      0.23
pole                 0.27        0.21      0.18
vegetation           0.34        0.28      0.17
traffic sign         0.26        0.24      0.15
wall                 0.29        0.22      0.11
building             0.24        0.16      0.11
